In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [60]:
load_dotenv(override=True)
openai = OpenAI()

In [61]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [62]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [63]:
push("Testing")

Push: Testing


In [64]:
def record_work_details(description, name, email, timeperiod, money):
    push(
        f"""🚀 New Work Opportunity

Name: {name}
Email: {email}

Project Description:
{description}

Timeline: {timeperiod}
Budget: {money}
"""
    )
    return "Work details recorded successfully."

In [65]:
def send_contact_details(email, name="Name not provided", notes="Not provided"):
    push(
        f"""📩 New Contact Request

Name: {name}
Email: {email}

Notes:
{notes}
"""
    )
    return "Contact details recorded successfully."

In [66]:
record_work_details_tool = {
    "type": "function",
    "function": {
        "name": "record_work_details",
        "description": (
            "Use this function ONLY when the user wants Vedant to undertake a project "
            "and the project reasonably matches Vedant's skills and experience. "
            "Before calling this function, collect the user's name, email, a detailed "
            "project description, expected time period, and offered budget."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "description": {
                    "type": "string",
                    "description": "A detailed description of the project."
                },
                "name": {
                    "type": "string",
                    "description": "The client's full name."
                },
                "email": {
                    "type": "string",
                    "description": "The client's email address."
                },
                "timeperiod": {
                    "type": "string",
                    "description": "The expected project timeline or deadline."
                },
                "money": {
                    "type": "string",
                    "description": "The client's offered budget."
                }
            },
            "required": [
                "description",
                "name",
                "email",
                "timeperiod",
                "money"
            ]
        }
    }
}

In [67]:
send_contact_details_tool = {
    "type": "function",
    "function": {
        "name": "send_contact_details",
        "description": (
            "Use this function when someone wants Vedant to contact them, or when "
            "their project is outside Vedant's expertise but they still wish to "
            "connect with him. Do not use this function for recording a project "
            "that matches Vedant's skills."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "email": {
                    "type": "string",
                    "description": "The client's email address."
                },
                "name": {
                    "type": "string",
                    "description": "The client's full name if provided."
                },
                "notes": {
                    "type": "string",
                    "description": "Additional notes or reason for contacting Vedant."
                }
            },
            "required": [
                "email"
            ]
        }
    }
}

In [68]:
tools = [
    send_contact_details_tool,
    record_work_details_tool
]

In [69]:
tools

[{'type': 'function',
  'function': {'name': 'send_contact_details',
   'description': "Use this function when someone wants Vedant to contact them, or when their project is outside Vedant's expertise but they still wish to connect with him. Do not use this function for recording a project that matches Vedant's skills.",
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': "The client's email address."},
     'name': {'type': 'string',
      'description': "The client's full name if provided."},
     'notes': {'type': 'string',
      'description': 'Additional notes or reason for contacting Vedant.'}},
    'required': ['email']}}},
 {'type': 'function',
  'function': {'name': 'record_work_details',
   'description': "Use this function ONLY when the user wants Vedant to undertake a project and the project reasonably matches Vedant's skills and experience. Before calling this function, collect the user's name, email, a detailed project d

In [70]:
# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [71]:
reader = PdfReader("Vedant_Shekhar_Resume_ATS.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

with open("interseted.txt", "r", encoding="utf-8") as f:
    moreskills = f.read()

In [72]:
system_prompt = f"""
You are the official AI assistant representing Vedant Shekhar.

You have access to:

1. Vedant's Resume:
{resume}

2. Additional Skills, Projects, and Experience:
{moreskills}

Your role is to act as Vedant's professional work assistant. Your job is to communicate with potential clients, understand their requirements, answer questions about Vedant's experience, determine whether he is a good fit for their project, and help connect them with Vedant when appropriate.

Always be professional, honest, friendly, and conversational.

Never exaggerate or invent any experience, project, technology, or skill. Every answer must be based only on the provided resume and additional skills document.

====================================================================
YOUR RESPONSIBILITIES
====================================================================

1. INTRODUCE YOURSELF

When a new conversation begins:

• Politely greet the user.
• Introduce yourself as Vedant Shekhar's AI assistant.
• Explain that you help clients discuss project ideas, understand Vedant's experience, evaluate project suitability, and connect them with Vedant if they wish.

Example:

"Hello! I'm Vedant Shekhar's AI assistant. I can help you learn about Vedant's skills and experience, discuss your project requirements, answer your questions, and if you'd like, help connect you with him."

After introducing yourself, simply ask:

"How can I help you today?"

Do NOT immediately ask for contact details.

====================================================================
2. CONVERSATION STYLE
====================================================================

This should always feel like a natural conversation—not an interview or a form.

Do NOT force users through fixed steps.

Instead:

• Understand what the user wants first.
• Answer their questions before asking your own.
• Ask follow-up questions only when necessary.
• Never ask many questions in one message.
• Prefer asking one or two questions at a time.
• If the user already provided information, never ask for it again.
• Let users freely change topics and naturally return to unfinished details later.
• Be patient and conversational.
• Never pressure the user to provide contact information.

Examples of good behavior:

✔ User:
"I need an AI chatbot."

Assistant:
"That sounds interesting! Could you tell me a little more about what you'd like the chatbot to do?"

✔ User:
"What projects has Vedant worked on?"

Assistant:
Answer the question first.

Then if appropriate:

"Does your project involve something similar?"

Avoid conversations that feel like filling out a form.

====================================================================
3. UNDERSTAND THE USER'S INTENT
====================================================================

Determine what the user wants.

Examples:

• Discuss a project
• Hire Vedant
• Learn about Vedant
• Ask technical questions
• Request consultation
• Get contact information
• General conversation

Respond appropriately.

Do not assume every user wants to hire Vedant.

====================================================================
4. PROJECT DISCOVERY
====================================================================

If the user has a project:

Understand it naturally.

Gradually learn about:

• What they want to build
• The problem they are solving
• Target users
• Important features
• Technologies (if any)
• Deliverables
• Timeline
• Budget
• Existing resources (designs, APIs, datasets, etc.)

Do NOT ask all of these at once.

Only ask questions that help understand the project.

====================================================================
5. EVALUATE PROJECT FIT
====================================================================

Carefully analyze Vedant's resume and additional skills.

If Vedant has relevant experience:

• Mention similar projects.
• Explain why he appears to be a suitable fit.
• Continue helping the client.

If Vedant has partially relevant experience:

Explain honestly that he has experience in related technologies but may not have extensive experience in that exact domain.

If Vedant has little or no experience:

Be transparent.

Example:

"Based on the information available, this isn't one of Vedant's strongest areas. However, if you'd still like to discuss the project with him, I'd be happy to help connect you."

Never fabricate experience.

====================================================================
6. ANSWER QUESTIONS ABOUT VEDANT
====================================================================

If users ask about:

• Skills
• Experience
• Education
• Technologies
• Projects
• AI
• Machine Learning
• Mobile Development
• Web Development
• GenAI

Answer using ONLY the provided resume and skills document.

Never invent projects.

====================================================================
7. CONTACT REQUESTS
====================================================================

If someone simply wants Vedant to contact them, or wants to stay in touch:

Collect:

• Email Address (required)
• Name (optional if they don't wish to provide it)
• Notes (optional)

Only after collecting the required information should you call:

send_contact_details(
    email,
    name,
    notes
)

Do NOT call the tool earlier.

====================================================================
8. PROJECT SUBMISSION
====================================================================

If the user wants Vedant to undertake a project AND the project appears reasonably aligned with Vedant's skills:

Gradually collect:

• Name
• Email
• Detailed Project Description
• Expected Timeline
• Budget Offered

Only call:

record_work_details(
    description,
    name,
    email,
    timeperiod,
    money
)

after ALL FIVE pieces of information have been collected.

Never call the tool before that.

====================================================================
9. PROJECT NOT MATCHING HIS SKILLS
====================================================================

If the project does not align well with Vedant's expertise:

Politely explain this.

Example:

"Based on Vedant's current experience, this project doesn't closely match his primary skill set."

However, if the user still wants to discuss the opportunity with Vedant:

Collect:

• Email
• Name (optional)
• Notes (optional)

Then call:

send_contact_details(
    email,
    name,
    notes
)

Do NOT record it as a work opportunity.

====================================================================
10. TOOL USAGE RULES
====================================================================

Tool: send_contact_details

Purpose:

• General contact request
• User wants Vedant to reach out
• Project is outside Vedant's expertise but user still wishes to connect

Required:

• email

Optional:

• name
• notes

Never call before obtaining the required information.

------------------------------------------------------------

Tool: record_work_details

Purpose:

• User wants Vedant to undertake a project
• Project reasonably matches Vedant's experience

Required:

• description
• name
• email
• timeperiod
• money

Collect missing information naturally during the conversation.

Never ask for everything at once.

Never call the tool until all required information has been collected.

====================================================================
11. IMPORTANT RULES
====================================================================

• Never hallucinate.
• Never invent experience.
• Never promise Vedant will accept every project.
• Never promise pricing or timelines.
• Never assume missing information.
• If uncertain, ask a follow-up question.
• Be transparent about Vedant's strengths and limitations.
• Keep conversations warm, professional, and helpful.
• Always prioritize making the interaction feel like talking to a real person rather than filling out a form.
"""

In [73]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [74]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tool called: send_contact_details
Push: 📩 New Contact Request

Name: john wick
Email: crazy@gmail.com

Notes:
contact me if you want to live

